# Train ConvNeXt-Tiny nhận diện loài cây trên Kaggle

Notebook này chỉ huấn luyện và đánh giá nhánh **plant species classification**.

Chuẩn bị trước khi chạy:

1. Add Data Source chứa dataset `v1.3` vào notebook Kaggle.
2. Bật **GPU** trong `Settings -> Accelerator`.
3. Bật **Internet** để notebook có thể clone source code khi project chưa được Add Input và tải pretrained weights ở lần chạy đầu.

Nếu source code AgriAI đã có trong `/kaggle/input`, notebook sẽ dùng bản đó. Nếu chưa có, notebook tự clone đúng branch từ GitHub vào `/kaggle/working/AgriAI`.

## 1. Cấu hình lần train plant

In [ ]:
from pathlib import Path

TASK = "plant"
DATASET_VERSION = "v1.3"
EPOCHS = 12
BATCH_SIZE = 8
NUM_WORKERS = 2
FREEZE_EPOCHS = 3
WARMUP_EPOCHS = 3
HEAD_LR = 3e-4
BACKBONE_LR = 3e-5
WEIGHT_DECAY = 0.05
TOP_K_CONFUSIONS = 20
INCLUDE_LAST_CHECKPOINT_IN_ZIP = False

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
PROJECT_DIR = KAGGLE_WORKING / "AgriAI"
PROJECT_REPOSITORY_URL = "https://github.com/KayPham05/AgriAI.git"
PROJECT_BRANCH = "feature/Model_Plant_Species_Classification"
EXTRACT_ROOT = KAGGLE_WORKING / "extracted_dataset"

print(f"Task: {TASK}")
print(f"Epochs: {EPOCHS}")
print(f"Batch vật lý: {BATCH_SIZE}")
print(f"Batch hiệu dụng dự kiến: {BATCH_SIZE * 4}")

## 2. Kiểm tra GPU

In [ ]:
import subprocess
import sys

result = subprocess.run(
    ["nvidia-smi"],
    text=True,
    capture_output=True,
    check=False,
)
print(result.stdout or result.stderr)
if result.returncode != 0:
    raise RuntimeError("Kaggle chưa bật GPU. Chọn Settings → Accelerator → GPU.")

import torch

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch không nhận CUDA trong runtime Kaggle.")

properties = torch.cuda.get_device_properties(0)
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {properties.total_memory / 1024**3:.1f} GB")

## 3. Tìm project trong Kaggle Input hoặc clone từ GitHub

Notebook ưu tiên source code đã được Add Input. Nếu không tìm thấy `ai/train.py`, notebook tự clone branch đã cấu hình vào `/kaggle/working/AgriAI`.

In [ ]:
import os
import shutil
import subprocess


def find_project_roots(search_root: Path) -> list[Path]:
    candidates: list[Path] = []
    for train_file in search_root.rglob("train.py"):
        if train_file.parent.name != "ai":
            continue
        project_root = train_file.parent.parent
        if (project_root / "ai" / "evaluate.py").is_file():
            candidates.append(project_root)
    return sorted(set(candidates), key=lambda path: (len(path.parts), str(path)))


def copy_project(source_dir: Path, destination_dir: Path) -> None:
    if destination_dir.exists():
        shutil.rmtree(destination_dir)
    shutil.copytree(
        source_dir,
        destination_dir,
        ignore=shutil.ignore_patterns(
            ".git",
            ".venv",
            "__pycache__",
            "checkpoints",
            "outputs",
            "runs",
        ),
    )


def clone_project(destination_dir: Path) -> None:
    if destination_dir.exists():
        shutil.rmtree(destination_dir)

    command = [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        PROJECT_BRANCH,
        PROJECT_REPOSITORY_URL,
        str(destination_dir),
    ]
    result = subprocess.run(command, text=True, capture_output=True)
    if result.returncode != 0:
        details = (result.stderr or result.stdout).strip()
        raise RuntimeError(
            "Không tìm thấy project trong /kaggle/input và không thể clone từ GitHub. "
            "Hãy bật Internet trong Kaggle hoặc Add Input chứa source code AgriAI.\n"
            f"Chi tiết từ git: {details}"
        )


if (PROJECT_DIR / "ai" / "train.py").is_file():
    project_source = PROJECT_DIR
    print("Dùng lại project đã có tại:", PROJECT_DIR)
else:
    project_candidates = find_project_roots(KAGGLE_INPUT)
    if project_candidates:
        project_source = project_candidates[0]
        copy_project(project_source, PROJECT_DIR)
        print("Đã sao chép project từ Kaggle Input:", project_source)
    else:
        clone_project(PROJECT_DIR)
        project_source = f"{PROJECT_REPOSITORY_URL} ({PROJECT_BRANCH})"
        print("Đã clone project từ GitHub:", project_source)

required_files = [PROJECT_DIR / "ai" / "train.py", PROJECT_DIR / "ai" / "evaluate.py"]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Project không đầy đủ, thiếu: " + ", ".join(str(path) for path in missing_files)
    )

os.chdir(PROJECT_DIR)
print("PROJECT_DIR:", PROJECT_DIR)

## 4. Tự tìm dataset đã giải nén hoặc giải nén archive

In [ ]:
import shutil


def find_dataset_roots(search_root: Path) -> list[Path]:
    matches = []
    for current_root, directory_names, file_names in os.walk(search_root):
        directory_names[:] = [
            name
            for name in directory_names
            if name not in {".git", "__pycache__", "checkpoints", "outputs", "images"}
        ]
        current_path = Path(current_root)
        if current_path.name == "manifests" and {
            "train.csv",
            "val.csv",
            "test.csv",
        }.issubset(file_names):
            candidate = current_path.parent
            if (candidate / "images").is_dir():
                matches.append(candidate)
    return sorted(set(matches))


def extract_dataset_archive(archive_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    suffix = archive_path.suffix.lower()
    if suffix == ".zip":
        shutil.unpack_archive(archive_path, destination)
        return
    if suffix == ".rar":
        command = ["7z", "x", str(archive_path), f"-o{destination}", "-y"]
        subprocess.run(command, check=True)
        return
    raise ValueError(f"Archive không được hỗ trợ: {archive_path}")


dataset_candidates = find_dataset_roots(KAGGLE_INPUT)
if not dataset_candidates:
    archives = sorted(
        path
        for path in KAGGLE_INPUT.rglob("*")
        if path.is_file()
        and path.suffix.lower() in {".zip", ".rar"}
        and DATASET_VERSION.lower() in path.name.lower()
    )
    if not archives:
        raise FileNotFoundError(
            "Không tìm thấy dataset đã giải nén hoặc archive v1.3 trong /kaggle/input."
        )
    print(f"Đang giải nén: {archives[0]}")
    extract_dataset_archive(archives[0], EXTRACT_ROOT)
    dataset_candidates = find_dataset_roots(EXTRACT_ROOT)

if not dataset_candidates:
    raise FileNotFoundError("Đã giải nén nhưng không tìm thấy images/ và manifests/.")
if len(dataset_candidates) > 1:
    print("Tìm thấy nhiều dataset; ưu tiên thư mục tên v1.3:")
    for candidate in dataset_candidates:
        print(" -", candidate)

dataset_candidates.sort(
    key=lambda path: (path.name != DATASET_VERSION, len(path.parts))
)
DATASET_DIR = dataset_candidates[0]
os.environ["AGRIVISION_DATASET_DIR"] = str(DATASET_DIR)

print(f"Dataset: {DATASET_DIR}")
print(f"Images: {DATASET_DIR / 'images'}")
print(f"Manifests: {DATASET_DIR / 'manifests'}")

## 5. Cài các dependency còn thiếu mà không thay PyTorch CUDA của Kaggle

In [ ]:
import importlib.util

required_packages = {
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "PIL": "pillow",
    "yaml": "pyyaml",
    "tensorboard": "tensorboard",
}
missing_packages = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    print("Cài dependency thiếu:", missing_packages)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        check=True,
    )
else:
    print("Các dependency cần thiết đã có sẵn.")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Working directory: {Path.cwd()}")

## 6. Kiểm tra manifest và phân bố 10 loài cây

In [ ]:
from collections import Counter

from ai.configs.classification_tasks import get_task_config
from ai.data.dataset import load_manifest_splits

plant_task = get_task_config("plant")
splits, class_to_idx, idx_to_info = load_manifest_splits(
    DATASET_DIR,
    target_field=plant_task.target_field,
)

if len(class_to_idx) != plant_task.expected_num_classes:
    raise ValueError(
        f"Plant cần {plant_task.expected_num_classes} lớp, "
        f"dataset cung cấp {len(class_to_idx)} lớp."
    )

print("Class mapping:")
for label, index in class_to_idx.items():
    print(f"  {index:2d}: {label}")

for split_name, records in splits.items():
    counts = Counter(record["plant"] for record in records)
    print(f"\n{split_name}: {len(records):,} ảnh")
    print(dict(sorted(counts.items())))

sample_paths = [
    DATASET_DIR / "images" / record["image_path"]
    for record in splits["train"][:100]
]
missing_samples = [path for path in sample_paths if not path.is_file()]
if missing_samples:
    raise FileNotFoundError(f"Thiếu ảnh mẫu: {missing_samples[0]}")
print("\nKiểm tra đường dẫn ảnh mẫu: OK")

## 7. Augmentation dành riêng cho plant

Dataset v1.3 đã ở 224×224 nên notebook không upscale. `RandomResizedCrop` giúp model nhìn các vùng lá ở tỷ lệ khác nhau và giảm phụ thuộc vào padding/nền. Validation và test vẫn giữ nguyên để so sánh công bằng trên split cố định.

In [ ]:
from torchvision import transforms
from torchvision.transforms import InterpolationMode

from ai.configs import config
from ai.data import augmentations as augmentation_module
from ai.data.augmentations import IMAGENET_MEAN, IMAGENET_STD
from ai.data.image_preprocessing import PADDING_COLOR


def build_robust_plant_transforms(image_size: int = config.IMAGE_SIZE):
    return transforms.Compose(
        [
            transforms.RandomResizedCrop(
                image_size,
                scale=(0.72, 1.0),
                ratio=(0.75, 1.3333),
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            ),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.1),
            transforms.RandomRotation(
                degrees=20,
                interpolation=InterpolationMode.BILINEAR,
                fill=PADDING_COLOR,
            ),
            transforms.ColorJitter(
                brightness=0.25,
                contrast=0.25,
                saturation=0.25,
                hue=0.03,
            ),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            transforms.RandomErasing(
                p=0.1,
                scale=(0.02, 0.08),
                ratio=(0.5, 2.0),
                value="random",
            ),
        ]
    )


augmentation_module.get_train_transforms = build_robust_plant_transforms
print("Đã kích hoạt augmentation plant mạnh hơn trong runtime notebook.")

## 8. Kiểm tra DataLoader trước khi train

In [ ]:
from ai.data.dataset import create_dataloaders

train_loader, val_loader, test_loader, num_classes, _ = create_dataloaders(
    dataset_dir=DATASET_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    target_field="plant",
    label_map_path=plant_task.label_map_path,
)

images, targets = next(iter(train_loader))
print(f"Train batches: {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")
print(f"Test batches: {len(test_loader):,}")
print(f"Batch image shape: {tuple(images.shape)}")
print(f"Batch target shape: {tuple(targets.shape)}")
print(f"Classes: {num_classes}")

if tuple(images.shape[1:]) != (3, 224, 224):
    raise ValueError(f"Shape đầu vào không đúng: {tuple(images.shape)}")

## 9. Train plant

Cấu hình mặc định:

- ConvNeXt-Tiny pretrained ImageNet-1K.
- Freeze backbone 3 epoch đầu.
- Fine-tune toàn bộ từ epoch 4.
- AdamW, weight decay 0.05.
- Warmup 3 epoch + cosine decay.
- Chọn checkpoint tốt nhất theo validation macro-F1.

Nếu CUDA hết bộ nhớ, đổi `BATCH_SIZE = 4` ở bước 1 rồi chạy lại notebook từ đầu. Không thêm `--no-pretrained` vì mục tiêu là fine-tune pretrained model.

In [ ]:
from argparse import Namespace

from ai.train import run_training

training_args = Namespace(
    task="plant",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    freeze_epochs=FREEZE_EPOCHS,
    warmup_epochs=WARMUP_EPOCHS,
    head_lr=HEAD_LR,
    backbone_lr=BACKBONE_LR,
    weight_decay=WEIGHT_DECAY,
    no_pretrained=False,
)

print("Bắt đầu train task plant...")
best_checkpoint = run_training(training_args)
print(f"Checkpoint tốt nhất: {best_checkpoint}")

## 10. Đánh giá checkpoint tốt nhất trên test split

In [ ]:
from ai.evaluate import evaluate_task

metrics = evaluate_task("plant", checkpoint_path=best_checkpoint)
print("\nTest metrics:")
for metric_name, metric_value in metrics.items():
    print(f"  {metric_name}: {metric_value:.6f}")

## 11. Xem biểu đồ và báo cáo

In [ ]:
import json
from IPython.display import Image as DisplayImage, display

checkpoint_dir = PROJECT_DIR / "ai" / "checkpoints" / "plant"
output_dir = PROJECT_DIR / "ai" / "outputs" / "plant"

for path in sorted(checkpoint_dir.glob("*")):
    if path.is_file():
        print(f"CHECKPOINT {path.name}: {path.stat().st_size / 1024**2:.2f} MB")
for path in sorted(output_dir.glob("*")):
    if path.is_file():
        print(f"OUTPUT     {path.name}: {path.stat().st_size / 1024**2:.2f} MB")

metrics_path = output_dir / "test_metrics.json"
if metrics_path.is_file():
    report = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("\n", json.dumps(report["metrics"], ensure_ascii=False, indent=2))

for image_name in ("training_history.png", "test_confusion_matrix.png"):
    image_path = output_dir / image_name
    if image_path.is_file():
        display(DisplayImage(filename=str(image_path)))

## 12. Đóng gói kết quả plant để tải về

In [ ]:
import json
from datetime import datetime, timezone
from IPython.display import FileLink, display

export_dir = KAGGLE_WORKING / "plant_training_export"
if export_dir.exists():
    shutil.rmtree(export_dir)
(export_dir / "checkpoints").mkdir(parents=True)
(export_dir / "outputs").mkdir(parents=True)

best_path = checkpoint_dir / "best_convnext_tiny.pth"
if not best_path.is_file():
    raise FileNotFoundError(f"Thiếu checkpoint tốt nhất: {best_path}")
shutil.copy2(best_path, export_dir / "checkpoints" / best_path.name)

last_path = checkpoint_dir / "last_convnext_tiny.pth"
if INCLUDE_LAST_CHECKPOINT_IN_ZIP and last_path.is_file():
    shutil.copy2(last_path, export_dir / "checkpoints" / last_path.name)

label_map_path = checkpoint_dir / "class_to_idx.json"
if label_map_path.is_file():
    shutil.copy2(label_map_path, export_dir / "checkpoints" / label_map_path.name)

for output_path in output_dir.glob("*"):
    if output_path.is_file():
        shutil.copy2(output_path, export_dir / "outputs" / output_path.name)

run_config = {
    "task": "plant",
    "dataset_version": DATASET_VERSION,
    "epochs_requested": EPOCHS,
    "batch_size": BATCH_SIZE,
    "effective_batch_size": BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS,
    "freeze_epochs": FREEZE_EPOCHS,
    "warmup_epochs": WARMUP_EPOCHS,
    "head_lr": HEAD_LR,
    "backbone_lr": BACKBONE_LR,
    "weight_decay": WEIGHT_DECAY,
    "image_size": config.IMAGE_SIZE,
    "augmentation": "robust_plant_random_resized_crop",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
(export_dir / "run_config.json").write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

archive_base = KAGGLE_WORKING / "plant_training_results"
archive_path = Path(
    shutil.make_archive(str(archive_base), "zip", root_dir=export_dir)
)
print(f"Đã tạo: {archive_path}")
print(f"Dung lượng: {archive_path.stat().st_size / 1024**2:.2f} MB")
display(FileLink(str(archive_path)))

## 13. Tải kết quả

Tải file sau từ panel **Output** bên phải của Kaggle:

```text
/kaggle/working/plant_training_results.zip
```

File ZIP chứa:

- `checkpoints/best_convnext_tiny.pth`
- `checkpoints/class_to_idx.json`
- `outputs/training_history.json`
- `outputs/training_summary.json`
- `outputs/training_history.png`
- `outputs/test_metrics.json`
- `outputs/classification_report.txt`
- `outputs/test_confusion_matrix.png`
- `outputs/misclassified_samples.csv`
- `outputs/top_confusions.csv`
- `run_config.json`

Sau khi tải về local, đặt checkpoint vào:

```text
ai/checkpoints/plant/best_convnext_tiny.pth
```

Lệnh dự đoán kết hợp plant → disease:

```powershell
python -m ai.predict --image "D:\anh_la.jpg" --topk 3
```